In [1]:
import json
import re
import random
import numpy as np
import torch

from pathlib import Path
from collections import Counter
from torch.utils.data import random_split

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.15.0+cpu
Device: cpu


In [2]:
# Set project directory
PROJECT_DIR = Path(r"D:\IE4483_Sentiment Project")

# Dataset path
TRAIN_PATH = PROJECT_DIR / "train.json"

# Check whether the file exists
assert TRAIN_PATH.exists(), f"File not found: {TRAIN_PATH}"

print("Project directory:", PROJECT_DIR)
print("Training file:", TRAIN_PATH)

Project directory: D:\IE4483_Sentiment Project
Training file: D:\IE4483_Sentiment Project\train.json


In [3]:
# Load training data
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print("Total samples:", len(train_data))

# Display sample reviews
print("\nSample 1:")
print(train_data[0])

print("\nSample 2:")
print(train_data[1])

Total samples: 7401

Sample 1:
{'reviews': 'I bought this belt for my daughter in-law for Christmas and she loved it', 'sentiments': 1}

Sample 2:
{'reviews': 'The size was perfect and so was the color.  It looked just like it did on the web page:)', 'sentiments': 1}


In [4]:
# Calculate split sizes
n_total = len(train_data)

n_val = int(0.15 * n_total)
n_train = n_total - n_val

# Split dataset
generator = torch.Generator().manual_seed(SEED)

train_set, val_set = random_split(
    train_data,
    [n_train, n_val],
    generator=generator
)

# Recover original records
train_records = [
    train_data[i] for i in train_set.indices
]

val_records = [
    train_data[i] for i in val_set.indices
]

print("Training samples:", len(train_records))
print("Validation samples:", len(val_records))

Training samples: 6291
Validation samples: 1110


In [5]:
train_labels = [
    item["sentiments"] for item in train_records
]

val_labels = [
    item["sentiments"] for item in val_records
]

print("Training distribution:")
print(Counter(train_labels))

print("\nValidation distribution:")
print(Counter(val_labels))

Training distribution:
Counter({1: 5357, 0: 934})

Validation distribution:
Counter({1: 962, 0: 148})


In [6]:
def tokenize(text):

    # Convert to lowercase
    text = text.lower()

    # Normalize apostrophes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")

    # Handle contractions
    text = text.replace("n't", "nt")

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        "",
        text
    )

    # Reduce repeated characters
    text = re.sub(
        r"([a-z])\1{2,}",
        r"\1",
        text
    )

    # Tokenization
    tokens = re.findall(r"[a-z0-9]+|[!?]", text)

    return tokens

In [7]:
# Tokenize training reviews
train_tokens = [
    tokenize(item["reviews"])
    for item in train_records
]

# Tokenize validation reviews
val_tokens = [
    tokenize(item["reviews"])
    for item in val_records
]

# Display examples
print("Original review:")
print(train_records[0]["reviews"])

print("\nTokenized review:")
print(train_tokens[0])

Original review:
I purchased this dress to be the maid of honor in my best friends wedding.  It arrived quickly, looked great, and was an awesome price.  It would be hard to beat this great deal

Tokenized review:
['i', 'purchased', 'this', 'dress', 'to', 'be', 'the', 'maid', 'of', 'honor', 'in', 'my', 'best', 'friends', 'wedding', 'it', 'arrived', 'quickly', 'looked', 'great', 'and', 'was', 'an', 'awesome', 'price', 'it', 'would', 'be', 'hard', 'to', 'beat', 'this', 'great', 'deal']


In [8]:
# Maximum vocabulary size
MAX_VOCAB = 20000

# Count word frequency in training data
word_counter = Counter()

for tokens in train_tokens:
    word_counter.update(tokens)

# Build vocabulary
vocab = {
    "<pad>": 0,
    "<unk>": 1
}

# Add the most frequent words
most_common = word_counter.most_common(MAX_VOCAB - 2)

for word, count in most_common:
    vocab[word] = len(vocab)

# Vocabulary size
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print("Most common words:")
print(word_counter.most_common(10))

Vocabulary size: 13028
Most common words:
[('the', 16618), ('i', 15045), ('and', 11218), ('a', 9128), ('to', 7363), ('it', 7145), ('is', 5490), ('for', 5247), ('of', 4889), ('they', 4352)]


In [9]:
# Convert tokens into numerical IDs
def encode_tokens(tokens, vocab):

    return [
        vocab.get(token, vocab["<unk>"])
        for token in tokens
    ]


# Encode training and validation data
train_token_ids = [
    encode_tokens(tokens, vocab)
    for tokens in train_tokens
]

val_token_ids = [
    encode_tokens(tokens, vocab)
    for tokens in val_tokens
]

# Convert labels into tensors
y_train = torch.tensor(
    train_labels,
    dtype=torch.float32
)

y_val = torch.tensor(
    val_labels,
    dtype=torch.float32
)

# Display an example
print("Original tokens:")
print(train_tokens[0])

print("\nEncoded IDs:")
print(train_token_ids[0])

print("\nLabel:")
print(y_train[0])

Original tokens:
['i', 'purchased', 'this', 'dress', 'to', 'be', 'the', 'maid', 'of', 'honor', 'in', 'my', 'best', 'friends', 'wedding', 'it', 'arrived', 'quickly', 'looked', 'great', 'and', 'was', 'an', 'awesome', 'price', 'it', 'would', 'be', 'hard', 'to', 'beat', 'this', 'great', 'deal']

Encoded IDs:
[3, 148, 12, 168, 6, 35, 2, 4324, 10, 3744, 13, 14, 124, 704, 614, 7, 248, 297, 260, 29, 4, 20, 83, 461, 70, 7, 45, 35, 258, 6, 742, 12, 29, 335]

Label:
tensor(1.)


In [10]:
# Calculate sequence lengths
train_lengths = np.array([
    len(ids) for ids in train_token_ids
])

val_lengths = np.array([
    len(ids) for ids in val_token_ids
])

print("Training Review Length:")
print("Median:", np.median(train_lengths))
print("90th percentile:", np.percentile(train_lengths, 90))
print("95th percentile:", np.percentile(train_lengths, 95))
print("Maximum:", np.max(train_lengths))

print("\nValidation Review Length:")
print("Median:", np.median(val_lengths))
print("90th percentile:", np.percentile(val_lengths, 90))
print("95th percentile:", np.percentile(val_lengths, 95))
print("Maximum:", np.max(val_lengths))

Training Review Length:
Median: 44.0
90th percentile: 111.0
95th percentile: 145.0
Maximum: 1084

Validation Review Length:
Median: 42.0
90th percentile: 114.0
95th percentile: 164.0
Maximum: 783


In [11]:
# Compare different maximum sequence lengths
candidate_lengths = [120, 160, 192, 256]

print("Sequence Length Comparison")
print("-" * 75)

for max_len in candidate_lengths:

    train_truncated = np.sum(train_lengths > max_len)
    val_truncated = np.sum(val_lengths > max_len)

    train_percentage = train_truncated / len(train_lengths) * 100
    val_percentage = val_truncated / len(val_lengths) * 100

    print(f"\nMAX_LEN = {max_len}")

    print(
        f"Training: {train_truncated} reviews "
        f"({train_percentage:.2f}%) truncated"
    )

    print(
        f"Validation: {val_truncated} reviews "
        f"({val_percentage:.2f}%) truncated"
    )

Sequence Length Comparison
---------------------------------------------------------------------------

MAX_LEN = 120
Training: 513 reviews (8.15%) truncated
Validation: 94 reviews (8.47%) truncated

MAX_LEN = 160
Training: 239 reviews (3.80%) truncated
Validation: 58 reviews (5.23%) truncated

MAX_LEN = 192
Training: 154 reviews (2.45%) truncated
Validation: 38 reviews (3.42%) truncated

MAX_LEN = 256
Training: 76 reviews (1.21%) truncated
Validation: 15 reviews (1.35%) truncated


5. Sequence Padding

In [12]:
# Maximum sequence length
MAX_LEN = 192

# Padding and truncation
def pad_sequences(sequences, max_len):

    padded_sequences = []

    for seq in sequences:

        # Truncate long sequences
        if len(seq) > max_len:
            seq = seq[:max_len]

        # Pad short sequences
        else:
            seq = seq + [vocab["<pad>"]] * (max_len - len(seq))

        padded_sequences.append(seq)

    return torch.tensor(padded_sequences, dtype=torch.long)


# Apply padding
X_train = pad_sequences(train_token_ids, MAX_LEN)
X_val = pad_sequences(val_token_ids, MAX_LEN)

# Keep actual sequence lengths for later CNN pooling
train_seq_lengths = torch.tensor(
    [min(len(ids), MAX_LEN) for ids in train_token_ids]
)

val_seq_lengths = torch.tensor(
    [min(len(ids), MAX_LEN) for ids in val_token_ids]
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("\nExample padded sequence:")
print(X_train[0])

Training shape: torch.Size([6291, 192])
Validation shape: torch.Size([1110, 192])

Example padded sequence:
tensor([   3,  148,   12,  168,    6,   35,    2, 4324,   10, 3744,   13,   14,
         124,  704,  614,    7,  248,  297,  260,   29,    4,   20,   83,  461,
          70,    7,   45,   35,  258,    6,  742,   12,   29,  335,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0

In [13]:
import zipfile

# 1. Define GloVe file paths
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

GLOVE_ZIP_PATH = DATA_DIR / "glove.6B.zip"
GLOVE_PATH = DATA_DIR / "glove.6B.100d.txt"


# 2. Check whether ZIP file exists
if not GLOVE_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"GloVe ZIP file not found: {GLOVE_ZIP_PATH}"
    )


# 3. Extract GloVe 100D if not already extracted
if not GLOVE_PATH.exists():

    with zipfile.ZipFile(GLOVE_ZIP_PATH, "r") as zip_ref:

        zip_ref.extract(
            "glove.6B.100d.txt",
            path=DATA_DIR
        )

    print("GloVe 100D extracted successfully!")

else:
    print("GloVe 100D already exists. Skipping extraction.")


# 4. Display file information
print("\nGloVe File Information")
print("----------------------")
print("ZIP file:", GLOVE_ZIP_PATH)
print("TXT file:", GLOVE_PATH)
print("ZIP exists:", GLOVE_ZIP_PATH.exists())
print("TXT exists:", GLOVE_PATH.exists())

print(
    "File size:",
    round(GLOVE_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

GloVe 100D already exists. Skipping extraction.

GloVe File Information
----------------------
ZIP file: D:\IE4483_Sentiment Project\data\glove.6B.zip
TXT file: D:\IE4483_Sentiment Project\data\glove.6B.100d.txt
ZIP exists: True
TXT exists: True
File size: 331.04 MB


In [14]:
# GloVe embedding dimension
EMBEDDING_DIM = 100

# Initialize embedding matrix
embedding_matrix = np.random.normal(
    loc=0.0,
    scale=0.1,
    size=(vocab_size, EMBEDDING_DIM)
).astype(np.float32)

# Set padding vector to zero
embedding_matrix[vocab["<pad>"]] = np.zeros(EMBEDDING_DIM)

# Store matched words
matched_words = set()

# Load GloVe vectors
with open(GLOVE_PATH, "r", encoding="utf-8") as f:

    for line in f:

        values = line.strip().split()

        word = values[0]

        # Only process words in our vocabulary
        if word in vocab:

            vector = np.asarray(
                values[1:],
                dtype=np.float32
            )

            if len(vector) == EMBEDDING_DIM:

                embedding_matrix[vocab[word]] = vector

                matched_words.add(word)

print("GloVe loading completed!")

GloVe loading completed!


In [15]:
print("Embedding Matrix Shape:", embedding_matrix.shape)

print(
    "Padding vector is zero:",
    np.all(embedding_matrix[vocab["<pad>"]] == 0)
)

print("Contains NaN:", np.isnan(embedding_matrix).any())
print("Contains Inf:", np.isinf(embedding_matrix).any())

Embedding Matrix Shape: (13028, 100)
Padding vector is zero: True
Contains NaN: False
Contains Inf: False


In [16]:
# Calculate vocabulary coverage
total_words = vocab_size - 2  # Exclude <pad> and <unk>

matched_count = len(matched_words)

coverage = matched_count / total_words * 100

# Frequency-weighted coverage
total_occurrences = sum(word_counter.values())

matched_occurrences = sum(
    word_counter[word] for word in matched_words
)

frequency_coverage = (
    matched_occurrences / total_occurrences * 100
)

print("Total vocabulary words:", total_words)
print("Matched GloVe words:", matched_count)
print("Unmatched words:", total_words - matched_count)

print(f"\nVocabulary coverage: {coverage:.2f}%")
print(f"Frequency-weighted coverage: {frequency_coverage:.2f}%")

Total vocabulary words: 13026
Matched GloVe words: 11326
Unmatched words: 1700

Vocabulary coverage: 86.95%
Frequency-weighted coverage: 99.34%


7. TextCNN Model Architecture

In [17]:
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader

# Batch size
BATCH_SIZE = 64

# Create TensorDataset
train_dataset = TensorDataset(
    X_train,
    train_seq_lengths,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    val_seq_lengths,
    y_val
)

# Create DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 99
Validation batches: 18


In [18]:
class TextCNN(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        num_filters=100,
        kernel_sizes=(3, 4, 5),
        dropout=0.5
    ):

        super(TextCNN, self).__init__()

        # 1. Pretrained GloVe Embedding
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False,
            padding_idx=0
        )

        # 2. Store kernel sizes
        self.kernel_sizes = kernel_sizes

        # 3. Parallel CNN branches
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=EMBEDDING_DIM,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in self.kernel_sizes
        ])

        # 4. Dropout
        self.dropout = nn.Dropout(dropout)

        # 5. Fully Connected Layer
        self.fc = nn.Linear(
            num_filters * len(self.kernel_sizes),
            1
        )


    def forward(self, inputs, lengths):

        # Embedding Layer
        embeds = self.embedding(inputs)

        # Change dimensions:
        # [Batch, Sequence, Embedding]
        # -> [Batch, Embedding, Sequence]
        embeds = embeds.transpose(1, 2)

        pooled_outputs = []

        # Apply parallel CNN branches
        for conv, kernel_size in zip(
            self.convs,
            self.kernel_sizes
        ):

            # Convolution + ReLU
            conv_out = F.relu(conv(embeds))

            # Calculate valid convolution positions
            valid_lengths = (
                lengths - kernel_size + 1
            ).clamp(min=0)

            # Create padding mask
            positions = torch.arange(
                conv_out.size(2),
                device=inputs.device
            ).unsqueeze(0)

            mask = positions < valid_lengths.unsqueeze(1)

            # Ignore convolution windows containing padding
            conv_out = conv_out.masked_fill(
                ~mask.unsqueeze(1),
                float("-inf")
            )

            # Global Max Pooling
            pooled = conv_out.max(dim=2).values

            # Handle reviews shorter than the kernel size
            pooled = torch.where(
                (valid_lengths > 0).unsqueeze(1),
                pooled,
                torch.zeros_like(pooled)
            )

            pooled_outputs.append(pooled)

        # Concatenate CNN features
        features = torch.cat(
            pooled_outputs,
            dim=1
        )

        # Dropout + Fully Connected Layer
        logits = self.fc(
            self.dropout(features)
        ).squeeze(1)

        return logits

In [19]:
# Initialize TextCNN model
model = TextCNN(
    embedding_matrix=embedding_matrix,
    num_filters=100,
    kernel_sizes=(3, 4, 5),
    dropout=0.5
).to(device)

# Display model architecture
print(model)

TextCNN(
  (embedding): Embedding(13028, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(100, 100, kernel_size=(3,), stride=(1,))
    (1): Conv1d(100, 100, kernel_size=(4,), stride=(1,))
    (2): Conv1d(100, 100, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=300, out_features=1, bias=True)
)


In [20]:
# Get one batch from training data
sample_inputs, sample_lengths, sample_labels = next(
    iter(train_loader)
)

# Move inputs to device
sample_inputs = sample_inputs.to(device)
sample_lengths = sample_lengths.to(device)

# Test forward pass
model.eval()

with torch.no_grad():

    sample_outputs = model(
        sample_inputs,
        sample_lengths
    )

# Display dimensions
print("Input shape:", sample_inputs.shape)
print("Output shape:", sample_outputs.shape)

print("\nTotal parameters:",
      sum(p.numel() for p in model.parameters()))

print("Trainable parameters:",
      sum(
          p.numel()
          for p in model.parameters()
          if p.requires_grad
      ))

Input shape: torch.Size([64, 192])
Output shape: torch.Size([64])

Total parameters: 1423401
Trainable parameters: 1423401


8. Model Training

In [21]:
# Training hyperparameters

LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
NUM_EPOCHS = 20
PATIENCE = 4

# Loss function
criterion = nn.BCEWithLogitsLoss()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Training Configuration")
print("----------------------")
print("Learning Rate:", LEARNING_RATE)
print("Weight Decay:", WEIGHT_DECAY)
print("Batch Size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)
print("Early Stopping Patience:", PATIENCE)
print("Optimizer: AdamW")
print("Loss Function: BCEWithLogitsLoss")

Training Configuration
----------------------
Learning Rate: 0.001
Weight Decay: 0.0001
Batch Size: 64
Epochs: 20
Early Stopping Patience: 4
Optimizer: AdamW
Loss Function: BCEWithLogitsLoss


In [22]:
def run_epoch(loader, training=True):

    # Set model mode
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    # Classification statistics
    tp = 0
    tn = 0
    fp = 0
    fn = 0

    with torch.set_grad_enabled(training):

        for xb, lengths, yb in loader:

            # Move data to device
            xb = xb.to(device)
            lengths = lengths.to(device)
            yb = yb.to(device)

            # Forward pass
            logits = model(xb, lengths)

            # Calculate loss
            loss = criterion(logits, yb)

            # Backpropagation
            if training:

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

            # Calculate predictions
            preds = (logits >= 0).long()
            labels = yb.long()

            # Update loss
            batch_size = yb.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            # Update classification statistics
            tp += ((preds == 1) & (labels == 1)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()

    # Calculate metrics
    avg_loss = total_loss / total_samples

    accuracy = (tp + tn) / total_samples

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [23]:
# Training process

best_val_loss = float("inf")
epochs_without_improvement = 0

history = []

CHECKPOINT_PATH = PROJECT_DIR / "best_textcnn.pt"

for epoch in range(1, NUM_EPOCHS + 1):

    # Training
    train_metrics = run_epoch(
        train_loader,
        training=True
    )

    # Validation
    val_metrics = run_epoch(
        val_loader,
        training=False
    )

    # Record history
    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"]
    })

    # Print results
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Acc: {train_metrics['accuracy']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    # Save best model
    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": best_val_loss,
            "val_acc": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"]
        }, CHECKPOINT_PATH)

        print("  -> Best model saved!")

    else:

        epochs_without_improvement += 1

    # Early stopping
    if epochs_without_improvement >= PATIENCE:

        print("\nEarly stopping triggered.")

        break


print("\nTraining completed!")
print("Best validation loss:", best_val_loss)

Epoch 01/20 | Train Loss: 0.3927 | Train Acc: 0.8512 | Val Loss: 0.2828 | Val Acc: 0.8766 | Val F1: 0.9335
  -> Best model saved!
Epoch 02/20 | Train Loss: 0.2731 | Train Acc: 0.8830 | Val Loss: 0.1998 | Val Acc: 0.9153 | Val F1: 0.9531
  -> Best model saved!
Epoch 03/20 | Train Loss: 0.1914 | Train Acc: 0.9285 | Val Loss: 0.1820 | Val Acc: 0.9189 | Val F1: 0.9551
  -> Best model saved!
Epoch 04/20 | Train Loss: 0.1421 | Train Acc: 0.9479 | Val Loss: 0.1554 | Val Acc: 0.9333 | Val F1: 0.9621
  -> Best model saved!
Epoch 05/20 | Train Loss: 0.0995 | Train Acc: 0.9682 | Val Loss: 0.1568 | Val Acc: 0.9369 | Val F1: 0.9645
Epoch 06/20 | Train Loss: 0.0662 | Train Acc: 0.9814 | Val Loss: 0.1501 | Val Acc: 0.9378 | Val F1: 0.9646
  -> Best model saved!
Epoch 07/20 | Train Loss: 0.0426 | Train Acc: 0.9897 | Val Loss: 0.1573 | Val Acc: 0.9441 | Val F1: 0.9683
Epoch 08/20 | Train Loss: 0.0284 | Train Acc: 0.9930 | Val Loss: 0.1671 | Val Acc: 0.9396 | Val F1: 0.9657
Epoch 09/20 | Train Loss: 0.0

In [24]:
# Load best checkpoint

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Best Epoch:", checkpoint["epoch"])

print("Best Validation Loss:",
      checkpoint["val_loss"])

print("Validation Accuracy:",
      checkpoint["val_acc"])

print("Validation F1:",
      checkpoint["val_f1"])

Best Epoch: 6
Best Validation Loss: 0.15005378650652396
Validation Accuracy: 0.9378378378378378
Validation F1: 0.9645608628659476


In [25]:
# Display training history without plotting

print("Number of epochs:", len(history))

print("\nTraining History")
print("-" * 95)

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Train Acc':<15}"
    f"{'Val Acc':<15}"
    f"{'Val F1':<15}"
)

print("-" * 95)

for item in history:

    print(
        f"{item['epoch']:<8}"
        f"{item['train_loss']:<15.4f}"
        f"{item['val_loss']:<15.4f}"
        f"{item['train_acc']:<15.4f}"
        f"{item['val_acc']:<15.4f}"
        f"{item['val_f1']:<15.4f}"
    )

print("-" * 95)

# Find best epoch
best_epoch = min(
    history,
    key=lambda x: x["val_loss"]
)

print("\nBest Epoch:", best_epoch["epoch"])
print("Best Validation Loss:", best_epoch["val_loss"])
print("Best Validation Accuracy:", best_epoch["val_acc"])
print("Best Validation F1:", best_epoch["val_f1"])

Number of epochs: 10

Training History
-----------------------------------------------------------------------------------------------
Epoch   Train Loss     Val Loss       Train Acc      Val Acc        Val F1         
-----------------------------------------------------------------------------------------------
1       0.3927         0.2828         0.8512         0.8766         0.9335         
2       0.2731         0.1998         0.8830         0.9153         0.9531         
3       0.1914         0.1820         0.9285         0.9189         0.9551         
4       0.1421         0.1554         0.9479         0.9333         0.9621         
5       0.0995         0.1568         0.9682         0.9369         0.9645         
6       0.0662         0.1501         0.9814         0.9378         0.9646         
7       0.0426         0.1573         0.9897         0.9441         0.9683         
8       0.0284         0.1671         0.9930         0.9396         0.9657         
9       0.022

In [26]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)

# Set model to evaluation mode
model.eval()

all_preds = []
all_labels = []

# Make predictions
with torch.no_grad():

    for xb, lengths, yb in val_loader:

        xb = xb.to(device)
        lengths = lengths.to(device)

        logits = model(xb, lengths)

        preds = (logits >= 0).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.numpy().astype(int))


# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)

macro_f1 = f1_score(
    all_labels,
    all_preds,
    average="macro"
)

balanced_acc = balanced_accuracy_score(
    all_labels,
    all_preds
)

cm = confusion_matrix(
    all_labels,
    all_preds,
    labels=[0, 1]
)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        labels=[0, 1],
        target_names=["Negative", "Positive"],
        digits=4,
        zero_division=0
    )
)

print("Overall Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Balanced Accuracy:", round(balanced_acc, 4))

Confusion Matrix:
[[102  46]
 [ 23 939]]

Classification Report:
              precision    recall  f1-score   support

    Negative     0.8160    0.6892    0.7473       148
    Positive     0.9533    0.9761    0.9646       962

    accuracy                         0.9378      1110
   macro avg     0.8846    0.8326    0.8559      1110
weighted avg     0.9350    0.9378    0.9356      1110

Overall Accuracy: 0.9378
Macro F1: 0.8559
Balanced Accuracy: 0.8326
